# Réplica de Nienkötter et al. (2026) — modelo local de Sichuan

Paper: *Assessing Debris-Flow Susceptibility at Local and Global Scales* (Remote Sensing 18(9), 1442).
Datos: Zenodo 10.5281/zenodo.20034823 · Código: gitlab.com/ankidmr/multi-scale-debris-flow-susceptibility

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

Corre las celdas en orden. Si Colab se corta, vuelve a correr las celdas 1 a 5 y luego la de entrenamiento: retoma desde el último pliegue terminado.

## 1. Revisar la GPU y montar Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/tesis_replica_nienkotter'   # aquí quedan datos y modelos
DATA_DRIVE = f'{DRIVE_DIR}/data'
RESULTS_DRIVE = f'{DRIVE_DIR}/results'
DATA_LOCAL = '/content/data'          # copia en el disco de Colab (más rápido)
REPO = '/content/repo'
for d in [DATA_DRIVE, RESULTS_DRIVE, DATA_LOCAL]:
    os.makedirs(d, exist_ok=True)
print('Listo')

## 2. Descargar los datos de Sichuan a Drive (solo la primera vez, ~2.6 GB)
Revisa el md5 de cada archivo y lo renombra al nombre que espera el código. Si ya están y el md5 cuadra, no los vuelve a bajar.

In [ ]:
import hashlib, subprocess

FILES = {
    # nombre en Zenodo: (md5, nombre que espera el código)
    'sichuan_locations_train.csv':  ('95fd176736a0f17ebe18773ecea4ad1b', 'locations_train.csv'),
    'sichuan_locations_test.csv':   ('f6367eed02f7ce195eefe9a616d73ec3', 'locations_test.csv'),
    'sichuan_locations_test.hdf5':  ('d68979f5b8eab608ebeb855349e71363', 'locations_test.hdf5'),
    'sichuan_locations_train.hdf5': ('511311c2d78aa73757e4b8261e60440e', 'locations_train.hdf5'),
}

def md5(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

for zname, (expected, local_name) in FILES.items():
    dest = f'{DATA_DRIVE}/{local_name}'
    if os.path.exists(dest) and md5(dest) == expected:
        print(f'OK (ya estaba): {local_name}')
        continue
    url = f'https://zenodo.org/records/20034823/files/{zname}?download=1'
    print(f'Descargando {zname} ...')
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', dest, url], check=True)
    got = md5(dest)
    assert got == expected, f'md5 no coincide en {local_name}: {got}. Borra el archivo y corre la celda de nuevo.'
    print(f'OK: {local_name}')

## 3. Clonar el código y corregir lo que falta
- Crea los dos JSON que el código busca y Zenodo no trae.
- `train.py`: pliegues configurables, semilla fija (para poder retomar), salta pliegues ya terminados y no guarda el estado del optimizador (ahorra espacio en Drive).
- `dataloader.py`: carga el HDF5 en memoria (mucho más rápido).

In [ ]:
import re, json
if not os.path.exists(REPO):
    !git clone -q https://gitlab.com/ankidmr/multi-scale-debris-flow-susceptibility.git {REPO}
!pip install -q geopandas h5py torchinfo

SRC = f'{REPO}/debris_flow_susceptibility'

def patch(path, pairs, out=None):
    txt = open(path).read()
    for old, new in pairs:
        n = txt.count(old) if isinstance(old, str) else len(re.findall(old, txt))
        assert n > 0, f'No encontré en {os.path.basename(path)}: {old!r}'
        txt = txt.replace(old, new) if isinstance(old, str) else re.sub(old, new, txt)
    open(out or path, 'w').write(txt)

# train.py -> train_patched.py
patch(f'{SRC}/train.py', [
    ('import argparse', 'import argparse\nimport os'),
    ('k = 10', "k = int(os.environ.get('N_FOLDS', 10))"),
    ('random_state=None', 'random_state=42'),
    ('enumerate(splits.split(np.arange(len(train_dataset)), groups=block_ids, y=class_ids)):',
     "enumerate(splits.split(np.arange(len(train_dataset)), groups=block_ids, y=class_ids)):\n"
     "        if fold >= int(os.environ.get('MAX_FOLDS', k)):\n"
     "            break\n"
     "        if (result_dir / f'done_fold_{fold}').exists():\n"
     "            print(f'Fold {fold} ya terminado, se salta', flush=True)\n"
     "            continue"),
    ("print(f'Fold {fold}: Validation AUC {auc_result:.4f}')",
     "print(f'Fold {fold}: Validation AUC {auc_result:.4f}', flush=True)\n"
     "        (result_dir / f'done_fold_{fold}').touch()"),
    (re.compile(r"\n\s*'optimizer_state_dict': optimizer\.state_dict\(\),"), ''),
    ('if result_dir.exists():', 'if False:'),
], out=f'{SRC}/train_patched.py')

# dataloader.py: HDF5 en memoria (se hace una sola vez; idempotente)
dl = open(f'{SRC}/dataloader.py').read()
if "['data'][:]}" not in dl:
    patch(f'{SRC}/dataloader.py', [
        ("self.hdf5file = h5py.File(hdf5_file, 'r')",
         "self.hdf5file = {'data': h5py.File(hdf5_file, 'r')['data'][:]}"),
    ])

# JSON con las 6 capas (el código solo usa la cantidad)
LAYERS = ['dem', 'soil_moisture', 'soil_depth', 'vegetation_index', 'max_precipitation', 'topsoil_clay']
for split in ['train', 'test']:
    json.dump(LAYERS, open(f'{DATA_DRIVE}/locations_{split}.json', 'w'))
print('Código listo')

## 4. Copiar los datos al disco de Colab y revisarlos
Se corre al inicio de cada sesión.

In [ ]:
import shutil, pandas as pd, h5py
for f in ['locations_train.csv', 'locations_test.csv', 'locations_train.json', 'locations_test.json',
          'locations_train.hdf5', 'locations_test.hdf5']:
    src, dst = f'{DATA_DRIVE}/{f}', f'{DATA_LOCAL}/{f}'
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        shutil.copy(src, dst)

for split in ['train', 'test']:
    df = pd.read_csv(f'{DATA_LOCAL}/locations_{split}.csv')
    with h5py.File(f'{DATA_LOCAL}/locations_{split}.hdf5', 'r') as h:
        shape, dtype = h['data'].shape, h['data'].dtype
    print(f'{split}: {len(df)} filas, hdf5 {shape} {dtype}')
    print(df['location type'].value_counts().to_dict(), '| grupos espaciales:', df['spatial_group_id'].nunique())

## 5. Prueba corta: 1 pliegue, 2 épocas
Mide cuánto tarda una época (mira la barra de `tqdm`). **Pásale a Claude el tiempo por época** para decidir cuántos pliegues caben.

In [ ]:
%cd {SRC}
import time
os.environ['N_FOLDS'] = '10'
os.environ['MAX_FOLDS'] = '1'
shutil.rmtree('/content/prueba', ignore_errors=True)
t0 = time.time()
!python train_patched.py --model ResNet --training-set {DATA_LOCAL} --save-folder /content/prueba --max-epochs 2 --hdf5-data
print(f'Tiempo total: {(time.time() - t0) / 60:.1f} min para 2 épocas')

## 6. Entrenar ResNet-50 (el modelo a igualar: AUC 0.957)
Máximo 40 épocas como en el paper. Cada pliegue terminado queda en Drive; si Colab se corta, corre las celdas 1–4 y esta de nuevo.

Si la prueba mostró que 10 pliegues no caben en tu tiempo de GPU, cambia `N_FOLDS` a 3 **antes de empezar** (no lo cambies a mitad: cambia la partición).

In [ ]:
N_FOLDS = 10
%cd {SRC}
os.environ['N_FOLDS'] = str(N_FOLDS)
os.environ.pop('MAX_FOLDS', None)
!python train_patched.py --model ResNet --training-set {DATA_LOCAL} --save-folder {RESULTS_DRIVE}/ResNet_k{N_FOLDS} --max-epochs 40 --hdf5-data

## 7. (Opcional) Entrenar VGG-16 (AUC del paper 0.949)
Ocupa más espacio por pliegue que ResNet-50.

In [ ]:
N_FOLDS = 10
%cd {SRC}
os.environ['N_FOLDS'] = str(N_FOLDS)
os.environ.pop('MAX_FOLDS', None)
!python train_patched.py --model VGG --training-set {DATA_LOCAL} --save-folder {RESULTS_DRIVE}/VGG_k{N_FOLDS} --max-epochs 40 --hdf5-data

## 8. Líneas base: Random Forest e Histogram Gradient Boosting (sin GPU)
El repositorio no trae estas líneas base y el paper no dice cómo armó sus entradas. Aquí cada parche se resume en 4 estadísticas por capa (media, desviación, mínimo, máximo = 24 variables). Es una **aproximación**: repórtala como tal.

In [ ]:
import numpy as np
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

def load_features(split):
    df = pd.read_csv(f'{DATA_LOCAL}/locations_{split}.csv')
    with h5py.File(f'{DATA_LOCAL}/locations_{split}.hdf5', 'r') as h:
        x = h['data'][:].reshape(len(df), 6, -1)
    X = np.concatenate([x.mean(2), x.std(2), x.min(2), x.max(2)], axis=1)
    return X, df['debris flow'].values, df['spatial_group_id'].values

def mean_ci(v):
    v = np.asarray(v)
    half = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
    return v.mean(), half

Xtr, ytr, gtr = load_features('train')
Xte, yte, _ = load_features('test')
K = 10
skf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=42)
baseline = {}
for name, make in [('Random Forest', lambda: RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)),
                   ('Histogram Gradient Boost', lambda: HistGradientBoostingClassifier(random_state=42))]:
    aucs = []
    for tr_idx, _ in skf.split(Xtr, ytr, groups=gtr):
        m = make().fit(Xtr[tr_idx], ytr[tr_idx])
        aucs.append(roc_auc_score(yte, m.predict_proba(Xte)[:, 1]))
    baseline[name] = mean_ci(aucs)
    print(f'{name}: AUC {baseline[name][0]:.3f} ± {baseline[name][1]:.3f}')

## 9. Resultados: AUC, calibración y curvas ROC frente al paper
Evalúa el mejor modelo de cada pliegue en el 10% de prueba. Pásale a Claude la tabla que imprime.

In [ ]:
import sys, glob, torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_curve, roc_auc_score

def mean_ci(v):
    v = np.asarray(v)
    half = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
    return v.mean(), half
sys.path.insert(0, SRC)
from models.ResNet import ResNet
from models.VGG import VGG
from dataloader import DebrisFlowsPrecomputedHDF5
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_set = DebrisFlowsPrecomputedHDF5(device, Path(DATA_LOCAL) / 'locations_test.csv', Path(DATA_LOCAL) / 'locations_test.hdf5')
loader = torch.utils.data.DataLoader(test_set, batch_size=32)
CLASSES = {'ResNet': ResNet, 'VGG': VGG}

@torch.inference_mode()
def predict(model):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(torch.sigmoid(model(x)).cpu().numpy().ravel())
        ys.append(y.cpu().numpy().ravel())
    return np.concatenate(ys), np.concatenate(ps)

def evaluate(run_dir, model_name):
    aucs, slopes, inters, rocs = [], [], [], []
    for ck in sorted(glob.glob(f'{run_dir}/all_layers/best_auc_fold_*.pth')):
        fold = ck.split('_')[-1].split('.')[0]
        if not os.path.exists(f'{run_dir}/all_layers/done_fold_{fold}'):
            continue
        d = torch.load(ck, map_location=device, weights_only=False)
        m = CLASSES[model_name](**d['model_params']).to(device)
        m.load_state_dict(d['model_state_dict'])
        y, p = predict(m)
        aucs.append(roc_auc_score(y, p))
        slope, inter = np.polyfit(p, y, 1)   # calibración: regresión lineal de (score, etiqueta)
        slopes.append(slope); inters.append(inter)
        rocs.append(roc_curve(y, p)[:2])
    return aucs, slopes, inters, rocs

PAPER = {'ResNet': (0.957, 0.002, 0.790, -0.134), 'VGG': (0.949, 0.005, 0.991, -0.132),
         'Random Forest': (0.874, 0.008), 'Histogram Gradient Boost': (0.894, 0.004)}
rows = []
plt.figure(figsize=(5, 5))
for run in sorted(glob.glob(f'{RESULTS_DRIVE}/*_k*')):
    name = os.path.basename(run).split('_k')[0]
    aucs, slopes, inters, rocs = evaluate(run, name)
    if len(aucs) < 2:
        print(f'{run}: {len(aucs)} pliegue(s) terminados, faltan más para el IC'); continue
    a, ah = mean_ci(aucs); s, sh = mean_ci(slopes); i, ih = mean_ci(inters)
    pa = PAPER[name]
    rows.append([name, len(aucs), f'{a:.3f} ± {ah:.3f}', f'{pa[0]:.3f} ± {pa[1]:.3f}',
                 f'{s:.3f} ± {sh:.3f}', pa[2], f'{i:.3f} ± {ih:.3f}', pa[3]])
    for fpr, tpr in rocs:
        plt.plot(fpr, tpr, alpha=0.3, label=None)
    plt.plot([], [], label=f'{name} (AUC {a:.3f})')
for name, (a, ah) in baseline.items() if 'baseline' in globals() else []:
    pa = PAPER[name]
    rows.append([name, K, f'{a:.3f} ± {ah:.3f}', f'{pa[0]:.3f} ± {pa[1]:.3f}', '', '', '', ''])
plt.plot([0, 1], [0, 1], 'k--', lw=0.8)
plt.xlabel('Tasa de falsos positivos'); plt.ylabel('Tasa de verdaderos positivos')
plt.title('Curvas ROC por pliegue — Sichuan (prueba)'); plt.legend(); plt.show()

res = pd.DataFrame(rows, columns=['Modelo', 'Pliegues', 'AUC (réplica)', 'AUC (paper)',
                                  'Pendiente calib. (réplica)', 'Pendiente (paper)',
                                  'Intercepto calib. (réplica)', 'Intercepto (paper)'])
display(res)
res.to_csv(f'{RESULTS_DRIVE}/tabla_resultados.csv', index=False)

## 10. AUC por pliegue y mediana (ResNet-50)
Requiere haber corrido la celda 9 en la misma sesión.

In [ ]:
aucs, _, _, _ = evaluate(f'{RESULTS_DRIVE}/ResNet_k10', 'ResNet')
print('AUC por pliegue:', np.round(aucs, 4))
print(f'Media {np.mean(aucs):.4f} | Mediana {np.median(aucs):.4f} | Mín {np.min(aucs):.4f} | Máx {np.max(aucs):.4f}')